# 4.0 Modelado

En este notebook se realiza de forma explícita todo el entrenamiento de los modelos de clustering. Así se puede observar y ejecutar paso a paso la selección de la población, la preparación de variables, PCA, K-Means, DBSCAN, las métricas y el guardado de resultados.

El análisis se concentra en estudiantes con bajo INSE. Los puntajes, percentiles, el INSE y la variable de resiliencia **no se usan para formar los clusters**; se conservan únicamente para interpretar los perfiles en la fase de evaluación.

## 1. Librerías, rutas y configuración

Se fija la semilla en 42 para que las selecciones aleatorias y los modelos entreguen los mismos resultados al volver a ejecutar el notebook.

In [ ]:
from pathlib import Path
import json
import os

# Un solo hilo evita pequeñas variaciones numéricas entre ejecuciones.
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import joblib
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
RARE_THRESHOLD = 0.005
OTHER_CATEGORY = 'OTRAS CATEGORÍAS'

project_root = Path.cwd() if Path('data').exists() else Path.cwd().parent
clean_path = project_root / 'data/processed/Examen_Saber_Pro_Genericas_2024_limpio.csv'
cluster_output_path = project_root / 'data/processed/saber_pro_2024_clusters_estudiantes.csv'
bundle_path = project_root / 'models/saber_pro_clustering_bundle.joblib'
summary_path = project_root / 'models/model_summary.json'

## 2. Variables usadas en el modelado

Las variables ordinales tienen un orden claro; las dicotómicas representan valores `Si` y `No`; y las categóricas no tienen un orden numérico natural. Las variables de perfil solo se guardan para describir posteriormente los grupos.

In [ ]:
ORDINAL_COLUMNS = [
    'fami_estratovivienda',
    'fami_educacionmadre',
    'fami_educacionpadre',
]

BINARY_COLUMNS = [
    'fami_tieneautomovil',
    'fami_tienecomputador',
    'fami_tienehornomicroogas',
    'fami_tieneinternet',
    'fami_tienelavadora',
    'fami_tienemotocicleta',
    'fami_tieneserviciotv',
]

CATEGORICAL_COLUMNS = [
    'fami_ocupacionmadre',
    'fami_ocupacionpadre',
    'fami_trabajolabormadre',
    'fami_trabajolaborpadre',
    'estu_tituloobtenidobachiller',
    'estu_actividadrefuerzogeneric',
    'estu_actividadrefuerzoareas',
    'estu_cursoiesexterna',
    'estu_cursoiesapoyoexterno',
    'estu_cursodocentesies',
    'estu_simulacrotipoicfes',
]

PROFILING_COLUMNS = [
    'estu_metodo_prgm',
    'estu_nivel_prgm_academico',
    'estu_nucleo_pregrado',
    'inst_caracter_academico',
    'inst_origen',
    'estu_prgm_departamento',
    'estu_prgm_academico',
    'estu_snies_prgmacademico',
]

MODEL_INPUT_COLUMNS = ORDINAL_COLUMNS + BINARY_COLUMNS + CATEGORICAL_COLUMNS

ESTRATO_MAP = {'Sin Estrato': 0, 'Sin estrato': 0}
ESTRATO_MAP.update({f'Estrato {number}': number for number in range(1, 7)})

EDUCATION_ORDER = [
    'Ninguno',
    'Primaria incompleta',
    'Primaria completa',
    'Secundaria (Bachillerato) incompleta',
    'Secundaria (Bachillerato) completa',
    'Técnica o tecnológica incompleta',
    'Técnica o tecnológica completa',
    'Educación profesional incompleta',
    'Educación profesional completa',
    'Postgrado',
]
EDUCATION_MAP = {value: position for position, value in enumerate(EDUCATION_ORDER)}
EDUCATION_MAP.update({'No sabe': 0, 'No Aplica': 0})

## 3. Lectura y selección de la población

Se conservan estudiantes con información socioeconómica y resultados válidos. El bajo INSE se define como el 25% inferior de la distribución válida. Se marca como resiliente a quien pertenece a esta población y alcanza un percentil global igual o superior a 90.

In [ ]:
required_columns = list(dict.fromkeys(
    [
        'estu_consecutivo',
        'estu_estudiante',
        'estado_contexto_socioeconomico',
        'punt_global',
        'percentil_global',
        'estu_inse_individual',
    ]
    + MODEL_INPUT_COLUMNS
    + PROFILING_COLUMNS
))

df = pd.read_csv(
    clean_path,
    usecols=required_columns,
    encoding='utf-8-sig',
    low_memory=False,
)

valid_rows = (
    df['estu_estudiante'].eq('ESTUDIANTE')
    & df['estado_contexto_socioeconomico'].eq('DISPONIBLE')
    & df['punt_global'].ne(-1)
    & df['percentil_global'].ne(-1)
)
valid_data = df.loc[valid_rows].copy()
inse_threshold = float(valid_data['estu_inse_individual'].quantile(0.25))

df_model = valid_data.loc[
    valid_data['estu_inse_individual'] <= inse_threshold
].copy().reset_index(drop=True)
df_model['resiliencia_flag'] = df_model['percentil_global'] >= 90

print(f'Estudiantes modelados: {len(df_model):,}')
print(f'Umbral de bajo INSE: {inse_threshold:.2f}')
print(f"Estudiantes resilientes: {df_model['resiliencia_flag'].sum():,}")

## 4. Preparación de variables

Las variables ordinales se estandarizan para llevarlas a una escala comparable. Las dicotómicas se convierten en 0 y 1. Las categorías que representan menos del 0,5% se agrupan como `OTRAS CATEGORÍAS`, evitando columnas casi vacías.

In [ ]:
ordinal_data = pd.DataFrame({
    'estrato_ordinal': df_model['fami_estratovivienda'].map(ESTRATO_MAP),
    'educacion_madre_ordinal': df_model['fami_educacionmadre'].map(EDUCATION_MAP),
    'educacion_padre_ordinal': df_model['fami_educacionpadre'].map(EDUCATION_MAP),
})

if ordinal_data.isna().any().any():
    invalid_columns = ordinal_data.columns[ordinal_data.isna().any()].tolist()
    raise ValueError(f'Hay categorías ordinales desconocidas en: {invalid_columns}')

ordinal_scaler = StandardScaler()
ordinal_scaled = pd.DataFrame(
    ordinal_scaler.fit_transform(ordinal_data),
    columns=ordinal_data.columns,
    index=df_model.index,
)

binary_data = df_model[BINARY_COLUMNS].apply(
    lambda column: column.map({'Si': 1, 'No': 0})
)
if binary_data.isna().any().any():
    invalid_columns = binary_data.columns[binary_data.isna().any()].tolist()
    raise ValueError(f'Hay valores binarios desconocidos en: {invalid_columns}')
binary_data = binary_data.astype('float32')

In [ ]:
grouped_categories = df_model[CATEGORICAL_COLUMNS].astype('string').copy()
rare_categories = {}
known_categories = {}
rare_rows = []

for column in CATEGORICAL_COLUMNS:
    frequencies = grouped_categories[column].value_counts(normalize=True)
    rare_values = frequencies[frequencies < RARE_THRESHOLD].index.astype(str).tolist()
    rare_categories[column] = rare_values

    rare_mask = grouped_categories[column].isin(rare_values)
    affected_rows = int(rare_mask.sum())
    grouped_categories.loc[rare_mask, column] = OTHER_CATEGORY

    categories = sorted(grouped_categories[column].dropna().astype(str).unique().tolist())
    if OTHER_CATEGORY not in categories:
        categories.append(OTHER_CATEGORY)
    known_categories[column] = categories
    grouped_categories[column] = pd.Categorical(
        grouped_categories[column], categories=categories
    )

    rare_rows.append({
        'variable': column,
        'categorias_agrupadas': len(rare_values),
        'registros_afectados': affected_rows,
    })

rare_summary = pd.DataFrame(rare_rows)
one_hot_data = pd.get_dummies(
    grouped_categories, prefix=CATEGORICAL_COLUMNS, dtype='float32'
)
feature_frame = pd.concat(
    [ordinal_scaled, binary_data, one_hot_data], axis=1
).astype('float32')
model_matrix = feature_frame.to_numpy(dtype='float32')

feature_metadata = {
    'ordinal_scaler': ordinal_scaler,
    'rare_threshold': RARE_THRESHOLD,
    'rare_categories': rare_categories,
    'known_categories': known_categories,
    'feature_names': feature_frame.columns.tolist(),
    'model_input_columns': MODEL_INPUT_COLUMNS,
    'ordinal_columns': ORDINAL_COLUMNS,
    'binary_columns': BINARY_COLUMNS,
    'categorical_columns': CATEGORICAL_COLUMNS,
}

display(rare_summary.query('categorias_agrupadas > 0'))
print(f'Variables entregadas a PCA: {model_matrix.shape[1]}')

## 5. Reducción de dimensionalidad con PCA

PCA resume las 101 variables del modelo en un grupo más pequeño de componentes. Para conservar las métricas actuales se mantiene el criterio seleccionado anteriormente: explicar al menos el 70% de la varianza. La gráfica permite ver cuántos componentes son necesarios para llegar a ese porcentaje.

In [ ]:
def calculate_metrics(matrix, labels, sample_size=10_000):
    size = min(sample_size, len(matrix))
    return {
        'silhouette': float(silhouette_score(
            matrix, labels, sample_size=size, random_state=RANDOM_STATE
        )),
        'davies_bouldin': float(davies_bouldin_score(matrix, labels)),
        'calinski_harabasz': float(calinski_harabasz_score(matrix, labels)),
    }

# PCA completo para observar la varianza acumulada.
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(model_matrix)

variance_accumulated = np.cumsum(pca_full.explained_variance_ratio_)
n_components_70 = int(np.argmax(variance_accumulated >= 0.70) + 1)

print(f'Componentes necesarios para explicar al menos 70%: {n_components_70}')

plt.figure(figsize=(8, 4))
plt.plot(
    range(1, len(variance_accumulated) + 1),
    variance_accumulated,
    marker='o',
    markersize=3,
)
plt.axhline(0.70, color='red', linestyle='--', label='70% de varianza')
plt.axvline(n_components_70, color='gray', linestyle=':')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza acumulada explicada')
plt.title('PCA: varianza acumulada')
plt.legend()
plt.tight_layout()
plt.show()

# Se conserva el mismo PCA que produjo las métricas actuales.
pca_kmeans = PCA(n_components=0.70, random_state=RANDOM_STATE)
kmeans_matrix = pca_kmeans.fit_transform(model_matrix).astype('float32')

print(f'Espacio reducido para K-Means: {kmeans_matrix.shape}')

## 6. Selección y entrenamiento de K-Means

Se utiliza el método del codo: se calcula la inercia para varios valores de `k` y se identifica dónde su disminución deja de ser tan marcada. Después se entrena el modelo final con 30 inicializaciones para obtener una solución estable.

In [ ]:
# Método del codo: se calcula la inercia para varios valores de k.
k_range = range(2, 11)
inertias = []

for k in k_range:
    kmeans_candidate = KMeans(
        n_clusters=k,
        n_init=10,
        max_iter=500,
        random_state=RANDOM_STATE,
    )
    kmeans_candidate.fit(kmeans_matrix)
    inertias.append(kmeans_candidate.inertia_)
    print(f'k={k}: inercia={kmeans_candidate.inertia_:,.0f}')

# El mayor cambio de pendiente señala el codo de la curva.
inertia_changes = np.diff(inertias)
change_ratios = inertia_changes[:-1] / inertia_changes[1:]
k_final = list(k_range)[np.argmax(change_ratios) + 1]

print(f'Número de clusters seleccionado: k = {k_final}')

plt.figure(figsize=(7, 4.5))
plt.plot(list(k_range), inertias, marker='o')
plt.axvline(k_final, color='red', linestyle='--', label=f'k = {k_final}')
plt.title('Método del codo para K-Means')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.legend()
plt.tight_layout()
plt.show()

# Modelo final con más inicializaciones para obtener una solución estable.
kmeans = KMeans(
    n_clusters=k_final,
    n_init=30,
    max_iter=500,
    random_state=RANDOM_STATE,
)
df_model['cluster_kmeans'] = kmeans.fit_predict(kmeans_matrix)
kmeans_metrics = calculate_metrics(
    kmeans_matrix, df_model['cluster_kmeans'].to_numpy()
)

print('Tamaño de cada cluster:')
display(df_model['cluster_kmeans'].value_counts().sort_index().to_frame('estudiantes'))
print('Métricas de K-Means:')
display(pd.Series(kmeans_metrics, name='resultado').round(3).to_frame())

K-Means conserva cerca del 70% de la varianza en 10 componentes y el método del codo selecciona cuatro clusters.

## 7. Selección y entrenamiento de DBSCAN

DBSCAN trabaja con una muestra reproducible de 30.000 estudiantes. Se usan tres componentes PCA con `whiten=True` para que tengan escalas comparables. El parámetro `min_samples` se mantiene en 50 y `eps` se obtiene con el percentil 90 de la distancia al vecino número 50, que corresponde a la configuración de las métricas actuales.

In [ ]:
pca_dbscan = PCA(n_components=3, whiten=True, random_state=RANDOM_STATE)
dbscan_matrix_all = pca_dbscan.fit_transform(model_matrix).astype('float32')

random_generator = np.random.default_rng(RANDOM_STATE)
dbscan_index = random_generator.choice(
    len(dbscan_matrix_all),
    size=min(30_000, len(dbscan_matrix_all)),
    replace=False,
)
dbscan_sample = dbscan_matrix_all[dbscan_index]

min_samples_dbscan = 50
neighbors = NearestNeighbors(
    n_neighbors=min_samples_dbscan, n_jobs=-1
).fit(dbscan_sample)
neighbor_distances = neighbors.kneighbors(dbscan_sample)[0][:, -1]
dbscan_eps = float(np.quantile(neighbor_distances, 0.90))

print(f'Muestra usada por DBSCAN: {len(dbscan_sample):,} estudiantes')
print(f'min_samples seleccionado: {min_samples_dbscan}')
print(f'eps seleccionado: {dbscan_eps:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(np.sort(neighbor_distances), color='#2563eb')
plt.axhline(dbscan_eps, color='red', linestyle='--', label=f'eps = {dbscan_eps:.3f}')
plt.xlabel('Estudiantes ordenados por distancia')
plt.ylabel('Distancia al vecino número 50')
plt.title('Selección de eps para DBSCAN')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
dbscan = DBSCAN(
    eps=dbscan_eps,
    min_samples=min_samples_dbscan,
    n_jobs=-1,
)
dbscan_labels = dbscan.fit_predict(dbscan_sample)

df_model['cluster_dbscan'] = np.nan
df_model.loc[dbscan_index, 'cluster_dbscan'] = dbscan_labels

# Se excluye el ruido (-1) para calcular las métricas de los clusters.
canonical_mask = (
    df_model['cluster_dbscan'].notna()
    & df_model['cluster_dbscan'].ne(-1)
)
dbscan_metrics = calculate_metrics(
    dbscan_matrix_all[canonical_mask.to_numpy()],
    df_model.loc[canonical_mask, 'cluster_dbscan'].astype(int).to_numpy(),
)

dbscan_cluster_count = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
dbscan_noise_rate = float((dbscan_labels == -1).mean() * 100)

print(f'Clusters encontrados: {dbscan_cluster_count}')
print(f'Registros considerados ruido: {dbscan_noise_rate:.2f}%')
print('Tamaño de cada cluster, incluyendo el ruido (-1):')
display(pd.Series(dbscan_labels).value_counts().sort_index().to_frame('estudiantes'))
print('Métricas de DBSCAN:')
display(pd.Series(dbscan_metrics, name='resultado').round(3).to_frame())

## 8. Construcción de perfiles

Después del entrenamiento se calculan resúmenes descriptivos para cada cluster. Estos cálculos no modifican los modelos; permiten interpretar qué caracteriza a cada grupo y estudiar en cuáles se concentra la resiliencia académica.

In [ ]:
df_model['indice_bienes_hogar'] = (
    df_model[BINARY_COLUMNS]
    .apply(lambda column: column.map({'Si': 1, 'No': 0}))
    .sum(axis=1)
)

def build_kmeans_profile(data):
    profile = data.groupby('cluster_kmeans').agg(
        estudiantes=('cluster_kmeans', 'size'),
        puntaje_promedio=('punt_global', 'mean'),
        percentil_promedio=('percentil_global', 'mean'),
        inse_promedio=('estu_inse_individual', 'mean'),
        bienes_promedio=('indice_bienes_hogar', 'mean'),
        resilientes=('resiliencia_flag', 'sum'),
        tasa_resiliencia=('resiliencia_flag', 'mean'),
    )
    profile['porcentaje_poblacion'] = profile['estudiantes'] / len(data) * 100
    profile['tasa_resiliencia'] *= 100
    total_resilient = max(int(data['resiliencia_flag'].sum()), 1)
    profile['participacion_resilientes'] = profile['resilientes'] / total_resilient * 100
    overall_rate = data['resiliencia_flag'].mean() * 100
    profile['indice_concentracion'] = profile['tasa_resiliencia'] / overall_rate
    return profile

def build_dbscan_profile(data):
    sample = data[data['cluster_dbscan'].notna()].copy()
    profile = sample.groupby('cluster_dbscan').agg(
        estudiantes=('cluster_dbscan', 'size'),
        resilientes=('resiliencia_flag', 'sum'),
        tasa_resiliencia=('resiliencia_flag', 'mean'),
        puntaje_promedio=('punt_global', 'mean'),
        inse_promedio=('estu_inse_individual', 'mean'),
        bienes_promedio=('indice_bienes_hogar', 'mean'),
    )
    profile['tasa_resiliencia'] *= 100
    return profile

def assign_cluster_names(data):
    clusters = sorted(data['cluster_kmeans'].dropna().astype(int).unique())
    names = {cluster: f'Perfil {cluster}' for cluster in clusters}

    preparation = data['estu_actividadrefuerzogeneric'].ne(
        'NO INFORMADO - BLOQUE NO DILIGENCIADO'
    )
    preparation_rate = preparation.groupby(data['cluster_kmeans']).mean()
    preparation_cluster = int(preparation_rate.idxmax())
    names[preparation_cluster] = 'Preparación académica reportada'

    remaining = [cluster for cluster in clusters if cluster != preparation_cluster]
    family_education = (
        data['fami_educacionmadre'].map(EDUCATION_MAP).fillna(0)
        + data['fami_educacionpadre'].map(EDUCATION_MAP).fillna(0)
    ) / 2
    education_mean = family_education.groupby(data['cluster_kmeans']).mean()
    education_cluster = int(education_mean.loc[remaining].idxmax())
    names[education_cluster] = 'Mayor capital educativo familiar'
    remaining.remove(education_cluster)

    goods_mean = data.groupby('cluster_kmeans')['indice_bienes_hogar'].mean()
    material_cluster = int(goods_mean.loc[remaining].idxmax())
    names[material_cluster] = 'Mayor disponibilidad material'
    remaining.remove(material_cluster)

    for cluster in remaining:
        names[cluster] = 'Mayor vulnerabilidad observada'
    return names

cluster_names = assign_cluster_names(df_model)
kmeans_profile = build_kmeans_profile(df_model).reset_index()
kmeans_profile['nombre'] = kmeans_profile['cluster_kmeans'].map(cluster_names)
dbscan_profile = build_dbscan_profile(df_model).reset_index()
display(kmeans_profile.round(2))

## 9. Guardado de datos y modelos

Se generan tres artefactos: un CSV con las etiquetas por estudiante, un bundle con las transformaciones y modelos entrenados, y un JSON con los indicadores del dashboard. Aunque la versión modular permanece en `src/models`, aquí los objetos se construyen y se guardan directamente para que el entrenamiento completo sea visible dentro del notebook.

In [ ]:
kmeans_columns = [f'pca_{number + 1}' for number in range(kmeans_matrix.shape[1])]
dbscan_columns = [f'dbscan_pca_{number + 1}' for number in range(3)]

kmeans_frame = pd.DataFrame(kmeans_matrix, columns=kmeans_columns)
kmeans_frame['pca_plot_1'] = kmeans_matrix[:, 0]
kmeans_frame['pca_plot_2'] = kmeans_matrix[:, 1]
dbscan_frame = pd.DataFrame(dbscan_matrix_all, columns=dbscan_columns)

export_columns = (
    [
        'estu_consecutivo', 'cluster_kmeans', 'cluster_dbscan',
        'resiliencia_flag', 'punt_global', 'percentil_global',
        'estu_inse_individual', 'indice_bienes_hogar',
    ]
    + ORDINAL_COLUMNS + BINARY_COLUMNS + CATEGORICAL_COLUMNS + PROFILING_COLUMNS
)
exported_data = pd.concat(
    [df_model[export_columns].reset_index(drop=True), kmeans_frame, dbscan_frame],
    axis=1,
)

model_info = {
    'inse_threshold': inse_threshold,
    'kmeans': {
        'clusters': int(kmeans.n_clusters),
        'pca_components': int(kmeans_matrix.shape[1]),
        'explained_variance': float(pca_kmeans.explained_variance_ratio_.sum()),
        'n_init': 30,
    },
    'dbscan': {
        'sample_size': int(len(dbscan_index)),
        'clusters': int(dbscan_cluster_count),
        'pca_components': 3,
        'eps': float(dbscan.eps),
        'min_samples': int(dbscan.min_samples),
        'noise_rate': dbscan_noise_rate,
    },
}

summary = {
    'schema_version': '1.0',
    'population': {
        'students': int(len(exported_data)),
        'resilient_students': int(exported_data['resiliencia_flag'].sum()),
        'resilience_rate': float(exported_data['resiliencia_flag'].mean() * 100),
        'average_score': float(exported_data['punt_global'].mean()),
        'inse_threshold': float(inse_threshold),
    },
    'kmeans': {
        **model_info['kmeans'],
        'metrics': kmeans_metrics,
        'cluster_names': {str(key): value for key, value in cluster_names.items()},
        'profiles': kmeans_profile.to_dict(orient='records'),
    },
    'dbscan': {
        **model_info['dbscan'],
        'metrics': dbscan_metrics,
        'profiles': dbscan_profile.to_dict(orient='records'),
    },
    'training': {
        'random_state': RANDOM_STATE,
        'rare_threshold': RARE_THRESHOLD,
        'feature_count': len(feature_metadata['feature_names']),
        'sklearn_version': sklearn.__version__,
        'source_file': clean_path.name,
    },
}

bundle = {
    'schema_version': '1.0',
    'random_state': RANDOM_STATE,
    'inse_threshold': inse_threshold,
    'feature_metadata': feature_metadata,
    'pca_kmeans': pca_kmeans,
    'kmeans': kmeans,
    'pca_dbscan': pca_dbscan,
    'dbscan': dbscan,
    'dbscan_training_indices': dbscan_index,
    'cluster_names': summary['kmeans']['cluster_names'],
    'metrics': {'kmeans': kmeans_metrics, 'dbscan': dbscan_metrics},
}

cluster_output_path.parent.mkdir(parents=True, exist_ok=True)
bundle_path.parent.mkdir(parents=True, exist_ok=True)
exported_data.to_csv(cluster_output_path, index=False, encoding='utf-8-sig')
joblib.dump(bundle, bundle_path, compress=3)
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('Archivos generados:')
print(f'- Datos con clusters: {cluster_output_path.resolve()}')
print(f'- Modelos entrenados: {bundle_path.resolve()}')
print(f'- Resumen del modelo: {summary_path.resolve()}')

## 10. Resultado del modelado

El notebook deja entrenados K-Means y DBSCAN con las mismas reglas usadas por el proyecto. También deja las etiquetas de cluster, los componentes PCA, las métricas internas y los objetos necesarios para reproducir las asignaciones de K-Means en el dashboard. La interpretación académica se desarrolla en `5.0-evaluation.ipynb`.